In [2]:
import pandas as pd
import duckdb
import matplotlib

df_raw = pd.read_excel('../data_files/brå_stockholm_21_25_raw.xls', header=None)

df_raw.head(10)


,0,1,2,3,4,5
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072
9,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN


In [4]:
# Döp om  kolumn '0' till 'Brottstyp'

df_cleaned = df_raw.rename(columns={0: 'Brottstyp'})

df_cleaned.head(10)

,Brottstyp,1,2,3,4,5
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072
9,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN


In [6]:
# Lägg till årtal i kolumn 1-5

years = [df_cleaned.iloc[3, i] for i in range(1, len(df_cleaned.columns))]

df_cleaned.columns = ['Brottstyp'] + years

df_cleaned.head(10)

,Brottstyp,2021,2022,2023,2024,2025
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072
9,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df_cleaned.dtypes)

Brottstyp       str
2021         object
2022         object
2023         object
2024         object
2025         object
dtype: object


In [8]:
# Justera data type för brottsdatan per capita (str -> int) 

year_cols = ['2021', '2022', '2023', '2024', '2025']
df_cleaned[year_cols] = df_cleaned[year_cols].apply(pd.to_numeric, errors='coerce')

In [9]:
print(df_cleaned.dtypes)

Brottstyp        str
2021         float64
2022         float64
2023         float64
2024         float64
2025         float64
dtype: object


In [18]:
list(df_cleaned[df_cleaned[['2021','2022','2023','2024','2025']].notna().any(axis=1)]['Brottstyp'])


[nan,
 'Misshandel inkl. grov (5, 6 §)',
 '4 kap. Brott mot frihet och frid',
 '6 kap. Sexualbrott',
 'Tillgrepp av motordrivet fortskaffningsmedel (7 §)',
 'Cykel',
 'Inbrottsstöld i bostad m.m.',
 'Från fordon m.m.',
 'Rån, inkl. grovt (5, 6 §)',
 '12 kap. Skadegörelsebrott',
 'Brott mot narkotikastrafflagen']

In [21]:
crime_types = [
    'Misshandel inkl. grov (5, 6 §)',
    '4 kap. Brott mot frihet och frid',
    '6 kap. Sexualbrott',
    'Tillgrepp av motordrivet fortskaffningsmedel (7 §)',
    'Cykel',
    'Från fordon m.m.',
    'Inbrottsstöld i bostad m.m.',
    'Rån, inkl. grovt (5, 6 §)',
    '12 kap. Skadegörelsebrott',
    'Brott mot narkotikastrafflagen'
]

In [23]:
df_filtered = df_cleaned[df_cleaned['Brottstyp'].isin(crime_types)]

In [25]:
df_cleaned = duckdb.sql("""--sql
SELECT
    CASE Brottstyp
        WHEN 'Misshandel inkl. grov (5, 6 §)' THEN 'Misshandel'
        WHEN '4 kap. Brott mot frihet och frid' THEN 'Hot & olaga intrång'
        WHEN '6 kap. Sexualbrott' THEN 'Sexualbrott'
        WHEN 'Tillgrepp av motordrivet fortskaffningsmedel (7 §)' THEN 'Bilstöld'
        WHEN 'Cykel' THEN 'Cykelstöld'
        WHEN 'Från fordon m.m.' THEN 'Stöld från fordon'
        WHEN 'Inbrottsstöld i bostad m.m.' THEN 'Bostadsinbrott'
        WHEN 'Rån, inkl. grovt (5, 6 §)' THEN 'Rån'
        WHEN '12 kap. Skadegörelsebrott' THEN 'Skadegörelse'
        WHEN 'Brott mot narkotikastrafflagen' THEN 'Narkotikabrott'
    END AS Brottstyp,
        * EXCLUDE (Brottstyp)              
FROM df_filtered
""").df()

In [26]:
df_cleaned.head(10)

,Brottstyp,2021,2022,2023,2024,2025
0,Misshandel,1004.0,978.0,1017.0,1039.0,1072.0
1,Hot & olaga intrång,1380.0,1309.0,1511.0,1756.0,1792.0
2,Sexualbrott,257.0,240.0,240.0,257.0,283.0
3,Bilstöld,191.0,171.0,138.0,122.0,85.0
4,Cykelstöld,940.0,699.0,645.0,653.0,580.0
5,Bostadsinbrott,485.0,399.0,376.0,397.0,360.0
6,Stöld från fordon,808.0,652.0,645.0,432.0,362.0
7,Rån,132.0,104.0,97.0,66.0,50.0
8,Skadegörelse,8819.0,7637.0,8939.0,7379.0,6208.0
9,Narkotikabrott,1462.0,1547.0,1537.0,1655.0,1642.0


In [27]:
df_cleaned.to_csv('../data_files/brottsstatistik_stockholm_cleaned.csv', index=False)

EDA - alla kommuner

In [30]:
df_sthlm_raw = pd.read_excel('../data_files/brå_alla_kommuner_2025_raw.xls', header=None)

df_sthlm_raw.head(20)

,0,1,2
0,Anmälda brott,NaN,Brottsförebyggande rådet
1,NaN,NaN,NaN
2,NaN,År,NaN
3,NaN,2025,NaN
4,NaN,/100 000 inv,NaN
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",603,NaN
9,3-7 kap. Brott mot person,NaN,NaN


In [36]:
# Döp om  kolumn '0' till 'Brottstyp'

df_sthlm_columns = df_sthlm_raw.rename(columns={0: 'Brottstyp'})

df_sthlm_columns.head(10)

,Brottstyp,1,2
0,Anmälda brott,NaN,Brottsförebyggande rådet
1,NaN,NaN,NaN
2,NaN,År,NaN
3,NaN,2025,NaN
4,NaN,/100 000 inv,NaN
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",603,NaN
9,3-7 kap. Brott mot person,NaN,NaN


In [37]:
df_sthlm_columns = df_sthlm_columns.rename(columns={1: '/100 000 inv'})

df_sthlm_columns.head(10)

,Brottstyp,/100 000 inv,2
0,Anmälda brott,NaN,Brottsförebyggande rådet
1,NaN,NaN,NaN
2,NaN,År,NaN
3,NaN,2025,NaN
4,NaN,/100 000 inv,NaN
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",603,NaN
9,3-7 kap. Brott mot person,NaN,NaN


In [55]:
# För att vi ska kunna filtrera på område

df_sthlm_columns['radnr'] = range(len(df_sthlm_columns))

df_sthlm_columns.head(100)

,Brottstyp,/100 000 inv,2,radnr
0,Anmälda brott,NaN,Brottsförebyggande rådet,0
1,NaN,NaN,NaN,1
2,NaN,År,NaN,2
3,NaN,2025,NaN,3
4,NaN,/100 000 inv,NaN,4
...,...,...,...,...
95,3-7 kap. Brott mot person,NaN,NaN,95
96,6 kap. Sexualbrott,279,NaN,96
97,8-12 kap. Brott mot förmögenhet,NaN,NaN,97
98,"8 kap. Stöld, rån m.m.",NaN,NaN,98


In [56]:
# Hitta vilka rader som gälller för resp. kommun

df_sthlm_columns[df_sthlm_columns['Brottstyp'].str.contains('Bromma|Enskede|Farsta|Hägersten|Hässelby|Järva|Kungsholmen|Norra|Skarpnäck|Skärholmen|Södermalm')]

,Brottstyp,/100 000 inv,2,radnr
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN,5
33,Stadsdelsområde Enskede - Årsta - Vantör (Sthlm),NaN,NaN,33
61,Stadsdelsområde Farsta (Sthlm),NaN,NaN,61
89,Stadsdelsområde Hägersten-Älvsjö (Sthlm),NaN,NaN,89
117,Stadsdelsområde Hässelby - Vällingby (Sthlm),NaN,NaN,117
145,Stadsdelsområde Järva (Sthlm),NaN,NaN,145
173,Stadsdelsområde Kungsholmen (Sthlm),NaN,NaN,173
201,Stadsdelsområde Norra innerstaden (Sthlm),NaN,NaN,201
229,Stadsdelsområde Skarpnäck (Sthlm),NaN,NaN,229
257,Stadsdelsområde Skärholmen (Sthlm),NaN,NaN,257


In [57]:
BROMMA_START = 5
ENSKEDE_START = 33
FARSTA_START = 61
ÄLVJSÖ_START = 89
HÄSSELBY_START = 117
JÄRVA_START = 145
KUNGSHOLMEN_START = 173
NORRA_INNERSTADEN_START = 201
SKARPNÄCK_START = 229
SKÄRHOLMEN_START = 257
SÖDERMALM_START = 285

df_cleaned = duckdb.sql(f"""--sql
SELECT 
    *,
    CASE 
        WHEN radnr >= {BROMMA_START}             AND radnr < {ENSKEDE_START}           THEN 'Bromma'
        WHEN radnr >= {ENSKEDE_START}            AND radnr < {FARSTA_START}            THEN 'Enskede - Årsta - Vantör'
        WHEN radnr >= {FARSTA_START}             AND radnr < {ÄLVJSÖ_START}            THEN 'Farsta' 
        WHEN radnr >= {ÄLVJSÖ_START}             AND radnr < {HÄSSELBY_START}          THEN 'Hägersten - Älvsjö'
        WHEN radnr >= {HÄSSELBY_START}           AND radnr < {JÄRVA_START}             THEN 'Hässelby - Vällingby'
        WHEN radnr >= {JÄRVA_START}              AND radnr < {KUNGSHOLMEN_START}       THEN 'Järva'
        WHEN radnr >= {KUNGSHOLMEN_START}        AND radnr < {NORRA_INNERSTADEN_START} THEN 'Kungsholmen'
        WHEN radnr >= {NORRA_INNERSTADEN_START}  AND radnr < {SKARPNÄCK_START}         THEN 'Norra innerstaden'
        WHEN radnr >= {SKARPNÄCK_START}          AND radnr < {SKÄRHOLMEN_START}        THEN 'Skarpnäck'
        WHEN radnr >= {SKÄRHOLMEN_START}         AND radnr < {SÖDERMALM_START}         THEN 'Skärholmen'
        WHEN radnr >= {SÖDERMALM_START}                                                THEN 'Södermalm'
    END AS Stadsdelsområde
FROM df_sthlm_columns
""").df()

df_cleaned.head(10)

,Brottstyp,/100 000 inv,2,radnr,Stadsdelsområde
0,Anmälda brott,NaN,Brottsförebyggande rådet,0,NaN
1,NaN,NaN,NaN,1,NaN
2,NaN,År,NaN,2,NaN
3,NaN,2025,NaN,3,NaN
4,NaN,/100 000 inv,NaN,4,NaN
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN,5,Bromma
6,3-7 kap. Brott mot person,NaN,NaN,6,Bromma
7,3 kap. Brott mot liv och hälsa,NaN,NaN,7,Bromma
8,"Misshandel inkl. grov (5, 6 §)",603,NaN,8,Bromma
9,3-7 kap. Brott mot person,NaN,NaN,9,Bromma


In [58]:
df_cleaned = df_cleaned.dropna(subset=['/100 000 inv'])

In [59]:
df_cleaned.head(10)

,Brottstyp,/100 000 inv,2,radnr,Stadsdelsområde
2,NaN,År,NaN,2,NaN
3,NaN,2025,NaN,3,NaN
4,NaN,/100 000 inv,NaN,4,NaN
8,"Misshandel inkl. grov (5, 6 §)",603,NaN,8,Bromma
10,4 kap. Brott mot frihet och frid,1372,NaN,10,Bromma
12,6 kap. Sexualbrott,253,NaN,12,Bromma
15,Tillgrepp av motordrivet fortskaffningsmedel (...,102,NaN,15,Bromma
19,Cykel,600,NaN,19,Bromma
22,"Stöld genom inbrott, inbrottsstöld i bostad, e...",646,NaN,22,Bromma
26,Från fordon m.m.,444,NaN,26,Bromma


In [60]:
crime_types = [
    'Misshandel inkl. grov (5, 6 §)',
    '4 kap. Brott mot frihet och frid',
    '6 kap. Sexualbrott',
    'Tillgrepp av motordrivet fortskaffningsmedel (7 §)',
    'Cykel',
    'Från fordon m.m.',
    'Inbrottsstöld i bostad m.m.',
    'Rån, inkl. grovt (5, 6 §)',
    '12 kap. Skadegörelsebrott',
    'Brott mot narkotikastrafflagen'
]

In [61]:
df_filtered = df_cleaned[df_cleaned['Brottstyp'].isin(crime_types)]

In [62]:
df_cleaned = duckdb.sql("""--sql
SELECT
    CASE Brottstyp
        WHEN 'Misshandel inkl. grov (5, 6 §)' THEN 'Misshandel'
        WHEN '4 kap. Brott mot frihet och frid' THEN 'Hot & olaga intrång'
        WHEN '6 kap. Sexualbrott' THEN 'Sexualbrott'
        WHEN 'Tillgrepp av motordrivet fortskaffningsmedel (7 §)' THEN 'Bilstöld'
        WHEN 'Cykel' THEN 'Cykelstöld'
        WHEN 'Från fordon m.m.' THEN 'Stöld från fordon'
        WHEN 'Inbrottsstöld i bostad m.m.' THEN 'Bostadsinbrott'
        WHEN 'Rån, inkl. grovt (5, 6 §)' THEN 'Rån'
        WHEN '12 kap. Skadegörelsebrott' THEN 'Skadegörelse'
        WHEN 'Brott mot narkotikastrafflagen' THEN 'Narkotikabrott'
    END AS Brottstyp,
        * EXCLUDE (Brottstyp)              
FROM df_filtered
""").df()

In [63]:
df_cleaned.head(50)

,Brottstyp,/100 000 inv,2,radnr,Stadsdelsområde
0,Misshandel,603,None,8,Bromma
1,Hot & olaga intrång,1372,None,10,Bromma
2,Sexualbrott,253,None,12,Bromma
3,Bilstöld,102,None,15,Bromma
4,Cykelstöld,600,None,19,Bromma
5,Stöld från fordon,444,None,26,Bromma
6,Rån,23,None,29,Bromma
7,Skadegörelse,3957,None,31,Bromma
8,Narkotikabrott,735,None,32,Bromma
9,Misshandel,1033,None,36,Enskede - Årsta - Vantör


In [64]:
df_cleaned = df_cleaned.drop(columns=['2', 'radnr'])

In [68]:
df_cleaned.head(40)

,Brottstyp,/100 000 inv,Stadsdelsområde
0,Misshandel,603,Bromma
1,Hot & olaga intrång,1372,Bromma
2,Sexualbrott,253,Bromma
3,Bilstöld,102,Bromma
4,Cykelstöld,600,Bromma
5,Stöld från fordon,444,Bromma
6,Rån,23,Bromma
7,Skadegörelse,3957,Bromma
8,Narkotikabrott,735,Bromma
9,Misshandel,1033,Enskede - Årsta - Vantör


In [70]:
df_cleaned.to_csv('../data_files/brå_alla_kommuner_2025_cleaned.csv', index=False)